In [ ]:
# text from https://www.reedbeta.com/blog/programmers-intro-to-unicode/
text = "Ｕｎｉｃｏｄｅ! 🅤🅝🅘🅒🅞🅓🅔‽ 🇺‌🇳‌🇮‌🇨‌🇴‌🇩‌🇪! 😄 The very name strikes fear and awe into the hearts of programmers worldwide. We all know we ought to “support Unicode” in our software (whatever that means—like using wchar_t for all the strings, right?). But Unicode can be abstruse, and diving into the thousand-page Unicode Standard plus its dozens of supplementary annexes, reports, and notes can be more than a little intimidating. I don’t blame programmers for still finding the whole thing mysterious, even 30 years after Unicode’s inception."
tokens = text.encode("utf-8") # raw bytes
tokens = list(map(int, tokens)) # convert to a list of integers in range 0..255 for convenience
print('---')
print(text)
print("length:", len(text))
print('---')
print(tokens)
print("length:", len(tokens))

In [ ]:
def get_stats(ids):
    counts = {}
    for pair in zip(ids, ids[1:]):
        counts[pair] = counts.get(pair, 0) + 1
    return counts

def merge(ids, max_pair, new_value):
    new_ids = []
    i = 0
    while i < len(ids):
        if i < len(ids) - 1 and (ids[i], ids[i+1]) == max_pair:
            new_ids.append(new_value)
            i += 2  # consume both elements of the pair
        else:
            new_ids.append(ids[i])
            i += 1
    return new_ids


vocab_size = 276 # 256 for byte values, plus 20 for the new merged tokens
new_merges = vocab_size - 256
merges = {}
for i in range(new_merges):
    stats = get_stats(tokens)
    top_pair = max(stats, key=stats.get)
    print(f"merging pair: {top_pair} with count {256 + i}")
    tokens = merge(tokens, top_pair, 256 + i)
    merges[top_pair] = 256 + i


In [29]:
vocab = {idx: bytes([idx]) for idx in range(256)}
for (po, p1), idx in merges.items():
    vocab[idx] = vocab[po] + vocab[p1]

def decode(tokens):
    tokens = b''.join(vocab[token] for token in tokens)
    text = tokens.decode("utf-8", errors="replace")
    return text
print(decode([104, 101, 108, 108, 111, 32, 119, 270, 108, 100, 33]))

hello world!


In [28]:
def encode(text):
    tokens = list(text.encode("utf-8"))

    for pair, value in merges.items():
        encoded_tokens = []
        i = 0
        while i < len(tokens):
            if i < len(tokens) - 1 and (tokens[i], tokens[i+1]) == pair:
                encoded_tokens.append(value)
                i += 2
            else:
                encoded_tokens.append(tokens[i])
                i += 1
        tokens = encoded_tokens
    return tokens

print(encode("hello world!"))


[104, 101, 108, 108, 111, 32, 119, 270, 108, 100, 33]


In [32]:
def encode(text):
    tokens = list(text.encode("utf-8"))
    while len(tokens) >= 2:
        stats = get_stats(tokens)
        pair = min(stats, key=lambda p: merges.get(p, float('inf')))
        if pair not in merges:
            break
        tokens = merge(tokens, pair, merges[pair])
    return tokens

print(encode("hello world!"))

[104, 101, 108, 108, 111, 32, 119, 270, 108, 100, 33]


In [34]:
print(decode(encode("hello worldnbtrvsfeagnhmj,mynthdgrsefa")))

hello worldnbtrvsfeagnhmj,mynthdgrsefa
